In [0]:
%pip install torch torchvision

In [0]:
import torch, mlflow, joblib, os
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import torchvision.models as models
import torch.nn as nn
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

import importlib
import config
importlib.reload(config)  # pick up config.py without restarting the kernel

from config import BATCH_SIZE, VARIANCE_CUTOFF, CORRELATION_THRESHOLD, K_BEST_FEATURES, get_run_id, VOLUME_FEATURES_DIR, DATASET_DIR
from dataset import OvarianDataset, inference_transforms

In [0]:
run_id = get_run_id()
print(run_id)

In [0]:
# Load data
splits_path = "/Volumes/compvis_od/data/ovarian_dataset/processed/splits.parquet"
df = pd.read_parquet(splits_path)

train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]

# Wrapping both train and validation into the class and batch chunks to feed in for the feature extraction
train_loader = DataLoader(
    OvarianDataset(train_df["path"].tolist(), train_df["label"].tolist(), inference_transforms),
    batch_size=BATCH_SIZE,
    shuffle=False,  
    num_workers=0
)

val_loader = DataLoader(
    OvarianDataset(val_df["path"].tolist(), val_df["label"].tolist(), inference_transforms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [0]:
weights = models.ResNet50_Weights.DEFAULT
resnet50 = models.resnet50(weights=weights)

resnet50.fc = nn.Identity()

for param in resnet50.parameters():
    param.requires_grad = False

resnet50.to(device)
resnet50.eval()

In [0]:
def extract_embeddings(data_loader, model, loader_name="extraction"):
    feature_list = []
    target_list = []
    num_batches = len(data_loader)

    with torch.no_grad():
        for i, (images, targets) in enumerate(data_loader, start=1):
            images = images.to(device)

            embeddings = model(images)

            feature_list.append(embeddings.cpu().numpy())
            target_list.append(targets.numpy())

            if i % 25 == 0 or i == num_batches:
                print(f"{loader_name}: batch {i}/{num_batches} done", flush=True)

    X = np.vstack(feature_list)
    y = np.concatenate(target_list)

    print(f"{loader_name}: complete - {X.shape[0]} images, {X.shape[1]} features")
    return X,y

In [0]:
X_train_features, y_train_labels = extract_embeddings(train_loader, resnet50)
print(f"Extracted Train Matrix Shape: {X_train_features.shape}")

In [0]:
X_val_features, y_val_labels = extract_embeddings(val_loader, resnet50)
print(f"Extracted Val Matrix Shape: {X_val_features.shape}")

In [0]:
var_selector = VarianceThreshold(threshold = VARIANCE_CUTOFF)
X_train_var = var_selector.fit_transform(X_train_features)
X_val_var = var_selector.transform(X_val_features)
print(X_train_var.shape[1])


df_var = pd.DataFrame(X_train_var)
corr_matrix = df_var.corr().abs()

upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > CORRELATION_THRESHOLD)]
non_colinear_indices = [col for col in df_var.columns if col not in to_drop]

X_train_non_corr = X_train_var[:, non_colinear_indices]
X_val_non_corr = X_val_var[:, non_colinear_indices]
print(X_train_non_corr.shape[1])


k_best_selector = SelectKBest(score_func = f_classif, k = K_BEST_FEATURES)
X_train_optimal = k_best_selector.fit_transform(X_train_non_corr, y_train_labels)
X_val_optimal = k_best_selector.transform(X_val_non_corr)
print(X_train_optimal.shape[1])


In [0]:
os.makedirs(VOLUME_FEATURES_DIR, exist_ok=True)


np.save(f"{VOLUME_FEATURES_DIR}/X_train_optimal.npy", X_train_optimal)
np.save(f"{VOLUME_FEATURES_DIR}/X_val_optimal.npy", X_val_optimal)
np.save(f"{VOLUME_FEATURES_DIR}/y_train_labels.npy", y_train_labels)
np.save(f"{VOLUME_FEATURES_DIR}/y_val_labels.npy", y_val_labels)

In [0]:
print(inference_transforms)

In [0]:
%pip install seaborn

In [0]:
X_train_optimal = np.load(f"{VOLUME_FEATURES_DIR}/X_train_optimal.npy")
y_train_labels = np.load(f"{VOLUME_FEATURES_DIR}/y_train_labels.npy")
class_names = sorted([d for d in os.listdir(DATASET_DIR) if not d.startswith(".")])

tsne = TSNE(n_components = 2, perplexity = 30, init = "pca", learning_rate = "auto", random_state = 14)
X_tsne = tsne.fit_transform(X_train_optimal)

tsne_df = pd.DataFrame({
    "tSNE_1": X_tsne[:, 0],
    "tSNE_2": X_tsne[:, 1],
    "Subtype": [class_names[idx] for idx in y_train_labels]
})

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=tsne_df,
    x="tSNE_1",
    y="tSNE_2",
    hue="Subtype",
    style="Subtype",
    palette="viridis",
    alpha=0.75,
    s=40
)
plt.title("t-SNE Projection of Top 200 Features", fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [0]:
with mlflow.start_run(run_id=run_id):
    
    mlflow.log_params({
        "batch_size": BATCH_SIZE,
        "variance_cutoff": VARIANCE_CUTOFF,
        "correlation_threshold": CORRELATION_THRESHOLD,
        "k_best_features": K_BEST_FEATURES,
        "image_resize_dim": "244x244",
        "norm_mean": "[0.485, 0.456, 0.406]",
        "norm_std": "[0.229, 0.224, 0.225]",
        "backbone": "resnet50"
    })
    
    # Log feature reduction metrics
    mlflow.log_metrics({
        "raw_features": X_train_features.shape[1],
        "features_after_variance": X_train_var.shape[1],
        "features_after_correlation": X_train_non_corr.shape[1],
        "final_k_features": X_train_optimal.shape[1],
        "train_samples": len(X_train_optimal),
        "val_samples": len(X_val_optimal)
    })

    artifacts_dir = "/tmp/feature_artifacts"
    os.makedirs(artifacts_dir, exist_ok=True)
    
    joblib.dump(var_selector, f"{artifacts_dir}/var_selector.pkl")
    joblib.dump(non_colinear_indices, f"{artifacts_dir}/non_colinear_indices.pkl")
    joblib.dump(k_best_selector, f"{artifacts_dir}/k_best_selector.pkl")
    
    mlflow.log_artifacts(artifacts_dir, artifact_path="feature_selectors")